# 12_02 · hidden-state 동결 풀링 프로브

`12_01`이 덤프한 exp1(8192)의 4종 풀링(mean·max·cls·last) 문서 벡터에, 동일한 **동결 선형 프로브**(val 특징 학습 → test 평가)를 걸어 test 길이 bin별 R-Precision·P@1·micro를 비교한다.

**핵심 질문**: mean 풀링이 장문에서 신호를 희석한다면 항희석 풀링(max)이 최장 bin(B3)에서 mean을 회수해야 한다. 비교는 상대적이다 — 모든 풀링이 동일한 선형 readout·동일 학습 설정을 쓰므로, 프로브 절대치가 파인튜닝 모델보다 낮은 것은 예상된 일(선형·val 학습 표본 한계)이고, 판정은 풀링 간 상대 순서(특히 B3에서 max vs mean)로 내린다.

SSOT = `output/hidden_pooling_probe_test.json`. 로컬 CPU 실행(GPU 불필요).

In [ ]:
import os, json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from dotenv import load_dotenv
from datasets import load_dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

from error_analysis import r_precision   # src/ editable 설치 최상위 모듈
from patent_train.metrics import sigmoid

load_dotenv()
ROOT = Path(os.environ["DATA_ROOT"])
os.environ["HF_HOME"] = str(ROOT / ".hf_cache")
OUT = ROOT / "output"

TAG = "modernbert-patent-len8192"
TOK_DS = "ingyoun/patent-clean-text-modernbert-tokenized"   # 12_01 덤프가 쓴 데이터셋(행순 동일)
NUM = 188
BINS = ["<=512", "512-1024", "1024-2048", ">2048"]
POOLS = ["mean", "max", "cls", "last"]
torch.manual_seed(42); np.random.seed(42)

## 라벨·길이축 + 히든 로드

덤프 행순은 토큰화 데이터셋 행순과 같다(`12_01`의 행순서 assert가 보증). 라벨(멀티핫)·길이 bin을 같은 데이터셋에서 읽고, 히든 4종 + concat(mean⊕max)을 로드한다.

In [ ]:
def load_axis(split):
    ds = load_dataset(TOK_DS, split=split)
    Y = np.asarray(ds["labels"], dtype=np.float32).astype(bool)   # (N,188) 멀티핫
    return Y, np.array(ds["length_bin"])

Yv, lbv = load_axis("val")
Yt, lbt = load_axis("test")
print(f"val {len(Yv):,}  test {len(Yt):,}  · test k>=2 {(Yt.sum(1)>=2).mean():.1%}")

def load_hidden(split, pool, n):
    arr = np.load(OUT / f"hidden_{TAG}_{split}_{pool}.npy").astype(np.float32)
    assert arr.shape == (n, 768), (split, pool, arr.shape)
    return arr

H = {"val": {}, "test": {}}
for p in POOLS:
    H["val"][p] = load_hidden("val", p, len(Yv))
    H["test"][p] = load_hidden("test", p, len(Yt))
H["val"]["mean+max"] = np.concatenate([H["val"]["mean"], H["val"]["max"]], 1)
H["test"]["mean+max"] = np.concatenate([H["test"]["mean"], H["test"]["max"]], 1)
FEATS = POOLS + ["mean+max"]
print("히든 4종 + concat 로드")

## 선형 프로브 — 동결 특징 위 linear readout

각 풀링 특징을 StandardScaler로 표준화하고(val fit) `Linear(d,188)`을 BCE로 학습한다(Adam, weight_decay=1e-3, 300 epoch — 전 풀링 동일). 랭킹 지표(R-Precision·P@1)는 threshold-free, micro는 val 튜닝 global τ.

In [ ]:
def train_probe(Xtr, Ytr, epochs=300, lr=1e-2, wd=1e-3):
    Xt = torch.tensor(Xtr); Yt_ = torch.tensor(Ytr.astype(np.float32))
    lin = nn.Linear(Xtr.shape[1], NUM); opt = torch.optim.Adam(lin.parameters(), lr=lr, weight_decay=wd)
    lossf = nn.BCEWithLogitsLoss(); lin.train()
    for _ in range(epochs):
        opt.zero_grad(); lossf(lin(Xt), Yt_).backward(); opt.step()
    lin.eval(); return lin

def eval_probe(lin, Xte):
    with torch.no_grad(): return lin(torch.tensor(Xte)).numpy()

def best_tau(scores_val, Yval):
    P = sigmoid(scores_val); best, bt = -1, 0.5
    for t in np.arange(0.1, 0.9, 0.02):
        f = f1_score(Yval, P >= t, average="micro", zero_division=0)
        if f > best: best, bt = f, float(t)
    return bt

def bin_metrics(scores, Y, lb, tau):
    P = sigmoid(scores); pred = P >= tau
    rp = r_precision(scores, Y); top1 = scores.argmax(1); p1 = Y[np.arange(len(Y)), top1]
    out = {}
    for b in ["ALL"] + BINS:
        sel = np.ones(len(Y), bool) if b == "ALL" else (lb == b)
        out[b] = {"n": int(sel.sum()),
                  "r_precision": round(float(rp[sel].mean()), 4),
                  "p_at_1": round(float(p1[sel].mean()), 4),
                  "micro_f1": round(float(f1_score(Y[sel], pred[sel], average="micro", zero_division=0)), 4)}
    return out

results = {}
for f in FEATS:
    sc = StandardScaler().fit(H["val"][f])
    Xv, Xt = sc.transform(H["val"][f]).astype(np.float32), sc.transform(H["test"][f]).astype(np.float32)
    lin = train_probe(Xv, Yv)
    tau = best_tau(eval_probe(lin, Xv), Yv)
    results[f] = {"tau": tau, "test": bin_metrics(eval_probe(lin, Xt), Yt, lbt, tau)}
    print(f"  {f:9s} done (τ={tau:.2f})")

## 결과 — 길이 bin별 R-Precision(주 지표)·P@1·micro

In [ ]:
def col(f, b, key): return results[f]["test"][b][key]

for title, key in [("R-Precision (threshold-free)", "r_precision"),
                   ("P@1 (threshold-free)", "p_at_1"),
                   ("micro-F1 (val 튜닝 global τ)", "micro_f1")]:
    print(f"\n=== {title} ===")
    print(f"{'pool':9s}" + "".join(f"{b:>12}" for b in ['ALL'] + BINS))
    for f in FEATS:
        print(f"{f:9s}" + "".join(f"{col(f,b,key):>12.4f}" for b in ['ALL'] + BINS))

## 판정 — 희석 가설 검정 + 저장

B3(>2048)에서 항희석 풀링이 mean을 회수하는지, concat이 상보 신호를 더하는지, 길이 기울기가 풀링과 무관한지로 판정한다.

In [ ]:
mean_b3 = col("mean", ">2048", "r_precision"); mean_all = col("mean", "ALL", "r_precision")
mean_slope = col("mean", "<=512", "r_precision") - mean_b3
print("B3 R-Prec Δ vs mean · ALL Δ · 길이기울기(mean {:.1f}pt):".format(mean_slope*100))
for f in FEATS:
    if f == "mean": continue
    dB3 = (col(f, ">2048", "r_precision") - mean_b3) * 100
    dALL = (col(f, "ALL", "r_precision") - mean_all) * 100
    sl = (col(f, "<=512", "r_precision") - col(f, ">2048", "r_precision")) * 100
    print(f"  {f:9s} B3 {dB3:+.2f}pt · ALL {dALL:+.2f}pt · 기울기 {sl:.1f}pt")

best_b3 = max(FEATS, key=lambda f: col(f, ">2048", "r_precision"))
gain_b3 = (col(best_b3, ">2048", "r_precision") - mean_b3) * 100
verdict = (
    f"B3 R-Precision 최고 풀링={best_b3}(mean 대비 {gain_b3:+.2f}pt). "
    + ("mean 최고/미달 → 항희석 풀링이 장문 신호를 회수 못함 → 희석 아티팩트 아님, 본질적 난이도 지지."
       if best_b3 == "mean" or gain_b3 < 0.5 else
       f"{best_b3}가 mean을 {gain_b3:.2f}pt 회수 → 풀링 희석 헤드룸 존재.")
)
print("\n" + verdict)

results["verdict"] = verdict
results["meta"] = {"tag": TAG, "n_val": len(Yv), "n_test": len(Yt),
                   "method": "frozen linear probe (val->test), StandardScaler, Adam wd=1e-3, 300ep",
                   "feats": FEATS, "bins": BINS}
fp = OUT / "hidden_pooling_probe_test.json"
fp.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", fp)